In [1]:
from __future__ import annotations

from dataclasses import dataclass
from enum import Enum
from pathlib import Path
from typing import Optional
from uuid import uuid4

In [2]:
EVENTS_MD_TEMPLATE = """# Events

Purpose:
This file is the canonical index of meaningful story events.

Definitions:
- Event:
  A bounded, world-relevant unit of change. An event is something that happens, is discovered, is decided, or is revealed in a way that meaningfully changes the story world, a character's knowledge, a relationship, a goal, a risk, or the consequences that follow. It is not merely a scene fragment or a sentence-level action. It should be stable enough to remain the same event even if later chapters refine its timing, participants, cause, or full meaning.

- UUID:
  A stable canonical identifier for the event. Its purpose is identity, not meaning. It allows the system to refer to the same event reliably across the index, the detailed event file, later updates, and agent workflows, even if the event's wording, timing, or interpretation evolves. UUIDs are generated programmatically and treated as the permanent handle for each event.

- When:
  The best currently known placement of the event in story time. Its purpose is to situate the event chronologically without forcing false precision. It may be exact, approximate, relative, inferred, or unknown, depending on what the manuscript currently supports. This field should remain open to later refinement as future chapters reveal stronger temporal grounding. It captures story-time placement, not document-edit time.

- Chapters:
  The manuscript locations where the event is evidenced, introduced, developed, or clarified. Its purpose is grounding and traceability. It tells the system and the reader where this event comes from in the text. It is an internal reference to the relevant chapter or chunk, not part of the event's meaning itself. Multiple chapters may be listed when an event is introduced in one place and clarified or extended later.

- Summary:
  A short canonical description of what the event is. Its purpose is to make the event legible at index level without retelling the full scene. It should describe the core happening in clear, neutral, world-level language, focusing on what changed or what became known. It should not drift into interpretation, prose style, emotional commentary, or speculation beyond what the manuscript supports.

Format:
- uuid | when | chapters | summary

Notes:
- one line per meaningful story event
- uuid is generated programmatically
- when may remain open if the manuscript does not yet support a stronger placement
- chapters may be a single chapter, multiple chapters, or a range
- summary should stay compact and canonical
- detailed event files live in ./story/events/<uuid>.md

## Entries
"""

In [3]:
class EventChangeType(str, Enum):
    ADD = "add"
    UPDATE = "update"
    REMOVE = "remove"


In [4]:
@dataclass(frozen=True, slots=True)
class EventRecord:
    uuid: str
    when_text: str
    chapters: str
    summary: str

    @classmethod
    def from_index_line(cls, line: str) -> "EventRecord":
        parts = [part.strip() for part in line.split("|", maxsplit=3)]
        if len(parts) != 4:
            raise ValueError(
                f"Invalid event line: {line!r}. "
                "Expected format: 'uuid | when | chapters | summary'."
            )

        event_uuid, when_text, chapters, summary = parts
        return cls(
            uuid=_validate_uuid(event_uuid),
            when_text=_normalize_index_value("when", when_text),
            chapters=_normalize_index_value("chapters", chapters),
            summary=_normalize_index_value("summary", summary),
        )

    def to_index_line(self) -> str:
        return f"{self.uuid} | {self.when_text} | {self.chapters} | {self.summary}"


In [5]:
@dataclass(frozen=True, slots=True)
class EventChange:
    change_type: EventChangeType
    proposed_record: Optional[EventRecord] = None

In [6]:
@dataclass(slots=True)
class TrackedEvent:
    uuid: str
    current_record: Optional[EventRecord] = None
    pending_change: Optional[EventChange] = None

    @property
    def exists_in_file(self) -> bool:
        return self.current_record is not None

    @property
    def has_pending_change(self) -> bool:
        return self.pending_change is not None

    @property
    def effective_record(self) -> Optional[EventRecord]:
        if self.pending_change is None:
            return self.current_record

        if self.pending_change.change_type == EventChangeType.REMOVE:
            return None

        return self.pending_change.proposed_record

    def stage_update(
        self,
        *,
        when_text: Optional[str] = None,
        chapters: Optional[str] = None,
        summary: Optional[str] = None,
    ) -> None:
        base_record = self.current_record
        if base_record is None and self.pending_change is not None:
            if self.pending_change.change_type in {
                EventChangeType.ADD,
                EventChangeType.UPDATE,
            }:
                base_record = self.pending_change.proposed_record

        if base_record is None:
            raise ValueError(f"Cannot update event {self.uuid!r} because it does not exist.")

        updated_record = EventRecord(
            uuid=self.uuid,
            when_text=_normalize_index_value("when", when_text, fallback=base_record.when_text),
            chapters=_normalize_index_value("chapters", chapters, fallback=base_record.chapters),
            summary=_normalize_index_value("summary", summary, fallback=base_record.summary),
        )

        if self.current_record is not None and updated_record == self.current_record:
            self.pending_change = None
            return

        change_type = EventChangeType.ADD if self.current_record is None else EventChangeType.UPDATE
        self.pending_change = EventChange(change_type=change_type, proposed_record=updated_record)

    def stage_removal(self) -> None:
        if self.current_record is None:
            if self.pending_change and self.pending_change.change_type == EventChangeType.ADD:
                self.pending_change = None
                return

            raise ValueError(f"Cannot remove event {self.uuid!r} because it does not exist.")

        self.pending_change = EventChange(change_type=EventChangeType.REMOVE)

    def merge_pending_change(self) -> Optional[EventChangeType]:
        if self.pending_change is None:
            return None

        change_to_merge = self.pending_change

        if change_to_merge.change_type == EventChangeType.REMOVE:
            self.current_record = None
        else:
            if change_to_merge.proposed_record is None:
                raise ValueError(
                    f"Pending change for event {self.uuid!r} is missing a proposed record."
                )
            self.current_record = change_to_merge.proposed_record

        self.pending_change = None
        return change_to_merge.change_type

In [7]:
@dataclass(frozen=True, slots=True)
class MergeResult:
    event_uuid: str
    merged: bool
    change_type: Optional[EventChangeType]
    message: str

In [8]:
class EventsAccessor:
    """
    Accessor layer over events.md.

    Public behavior:
    - reads events.md on initialization
    - exposes self.events as a dict[str, TrackedEvent]
    - can create, update, remove, and merge one event at a time
    - rewrites events.md from committed state whenever a pending change is merged
    """

    def __init__(self, events_md_path: str | Path) -> None:
        self.events_md_path = Path(events_md_path)
        self.event_records_directory = self.events_md_path.parent / self.events_md_path.stem
        self.events: dict[str, TrackedEvent] = {}

        self._ensure_file_exists()
        self._load_from_file()
        self._validate_indexed_event_files()

    def get_event(self, event_uuid: str) -> Optional[TrackedEvent]:
        return self.events.get(event_uuid)

    def create_event(self, *, when_text: str, chapters: str, summary: str) -> TrackedEvent:
        event_uuid = self._generate_event_uuid()

        new_record = EventRecord(
            uuid=event_uuid,
            when_text=_normalize_index_value("when", when_text),
            chapters=_normalize_index_value("chapters", chapters),
            summary=_normalize_index_value("summary", summary),
        )

        tracked_event = TrackedEvent(
            uuid=event_uuid,
            current_record=None,
            pending_change=EventChange(
                change_type=EventChangeType.ADD,
                proposed_record=new_record,
            ),
        )

        self.events[event_uuid] = tracked_event
        return tracked_event

    def update_event(
        self,
        event_uuid: str,
        *,
        when_text: Optional[str] = None,
        chapters: Optional[str] = None,
        summary: Optional[str] = None,
    ) -> TrackedEvent:
        tracked_event = self._require_event(event_uuid)
        tracked_event.stage_update(
            when_text=when_text,
            chapters=chapters,
            summary=summary,
        )
        return tracked_event

    def remove_event(self, event_uuid: str) -> Optional[TrackedEvent]:
        tracked_event = self._require_event(event_uuid)
        tracked_event.stage_removal()

        # Special case:
        # If this was an unmerged "add" and removal canceled it,
        # there is no reason to keep an empty shell around.
        if not tracked_event.exists_in_file and not tracked_event.has_pending_change:
            self.events.pop(event_uuid, None)
            return None

        return tracked_event

    def merge_pending_change(self, event_uuid: str) -> MergeResult:
        tracked_event = self._require_event(event_uuid)
        merged_change_type = tracked_event.merge_pending_change()

        if merged_change_type is None:
            return MergeResult(
                event_uuid=event_uuid,
                merged=False,
                change_type=None,
                message=f"No pending change to merge for event {event_uuid}.",
            )

        if tracked_event.current_record is None and not tracked_event.has_pending_change:
            self.events.pop(event_uuid, None)

        self._write_file()

        return MergeResult(
            event_uuid=event_uuid,
            merged=True,
            change_type=merged_change_type,
            message=f"Merged {merged_change_type.value} change for event {event_uuid}.",
        )

    def reload_from_file(self) -> None:
        self._load_from_file()
        self._validate_indexed_event_files()

    def list_events(self) -> list[TrackedEvent]:
        return list(self.events.values())

    def list_committed_events(self) -> list[EventRecord]:
        committed_events: list[EventRecord] = []
        for tracked_event in self.events.values():
            if tracked_event.current_record is not None:
                committed_events.append(tracked_event.current_record)
        return committed_events

    def _ensure_file_exists(self) -> None:
        self.events_md_path.parent.mkdir(parents=True, exist_ok=True)
        if not self.events_md_path.exists():
            self.events_md_path.write_text(EVENTS_MD_TEMPLATE, encoding="utf-8")

    def _load_from_file(self) -> None:
        file_text = self.events_md_path.read_text(encoding="utf-8")
        event_records = self._parse_events_md(file_text)

        self.events = {
            record.uuid: TrackedEvent(uuid=record.uuid, current_record=record)
            for record in event_records
        }

    def _parse_events_md(self, file_text: str) -> list[EventRecord]:
        marker = "## Entries"
        if marker not in file_text:
            raise ValueError(f"{self.events_md_path} is missing the '## Entries' section.")

        _, entries_block = file_text.split(marker, maxsplit=1)

        parsed_records: list[EventRecord] = []
        seen_uuids: set[str] = set()

        for raw_line in entries_block.splitlines():
            line = raw_line.strip()
            if not line:
                continue

            record = EventRecord.from_index_line(line)

            if record.uuid in seen_uuids:
                raise ValueError(f"Duplicate event UUID found in {self.events_md_path}: {record.uuid}")

            seen_uuids.add(record.uuid)
            parsed_records.append(record)

        return parsed_records

    def _write_file(self) -> None:
        committed_lines = [record.to_index_line() for record in self.list_committed_events()]
        rendered_content = EVENTS_MD_TEMPLATE + "\n".join(committed_lines)

        if committed_lines:
            rendered_content += "\n"

        self.events_md_path.write_text(rendered_content, encoding="utf-8")

    def _generate_event_uuid(self) -> str:
        while True:
            candidate_uuid = f"evt_{uuid4().hex[:12]}"
            if candidate_uuid not in self.events:
                return candidate_uuid

    def _require_event(self, event_uuid: str) -> TrackedEvent:
        tracked_event = self.get_event(event_uuid)
        if tracked_event is None:
            raise KeyError(f"Event {event_uuid!r} was not found.")
        return tracked_event

In [9]:
def _validate_uuid(value: str) -> str:
    cleaned_value = value.strip()
    if not cleaned_value:
        raise ValueError("UUID cannot be empty.")
    if "|" in cleaned_value or "\n" in cleaned_value or "\r" in cleaned_value:
        raise ValueError(f"UUID contains invalid characters: {value!r}")
    return cleaned_value


def _normalize_index_value(field_name: str, value: Optional[str], fallback: Optional[str] = None) -> str:
    candidate = fallback if value is None else value
    if candidate is None:
        raise ValueError(f"{field_name!r} is required.")

    cleaned_value = candidate.strip()
    if not cleaned_value:
        cleaned_value = "-"

    if "|" in cleaned_value:
        raise ValueError(
            f"{field_name!r} cannot contain '|', because events.md uses pipe-separated entries."
        )

    if "\n" in cleaned_value or "\r" in cleaned_value:
        raise ValueError(f"{field_name!r} must stay on a single line.")

    return cleaned_value

In [10]:
accessor = EventsAccessor("./story/events.md")

In [11]:
new_event = accessor.create_event(
    when_text="Night of the storm",
    chapters="Chapter 3",
    summary="Mira discovers the sealed letter in the chapel.",
)

In [12]:
print(new_event.uuid)                       # generated uuid
print(new_event.exists_in_file)            # False
print(new_event.pending_change) 

evt_af6599fe7a30
False
EventChange(change_type=<EventChangeType.ADD: 'add'>, proposed_record=EventRecord(uuid='evt_af6599fe7a30', when_text='Night of the storm', chapters='Chapter 3', summary='Mira discovers the sealed letter in the chapel.'))


In [13]:
result = accessor.merge_pending_change(new_event.uuid)
print(result.message)

Merged add change for event evt_af6599fe7a30.


In [14]:
accessor.update_event(
    new_event.uuid,
    chapters="Chapters 3-4",
    summary="Mira discovers the sealed letter and begins connecting it to the chapel.",
)

TrackedEvent(uuid='evt_af6599fe7a30', current_record=EventRecord(uuid='evt_af6599fe7a30', when_text='Night of the storm', chapters='Chapter 3', summary='Mira discovers the sealed letter in the chapel.'), pending_change=EventChange(change_type=<EventChangeType.UPDATE: 'update'>, proposed_record=EventRecord(uuid='evt_af6599fe7a30', when_text='Night of the storm', chapters='Chapters 3-4', summary='Mira discovers the sealed letter and begins connecting it to the chapel.')))

In [15]:
result = accessor.merge_pending_change(new_event.uuid)
print(result.message)


Merged update change for event evt_af6599fe7a30.


In [16]:
accessor.remove_event(new_event.uuid)
result = accessor.merge_pending_change(new_event.uuid)
print(result.message)

Merged remove change for event evt_af6599fe7a30.
